In [1]:
from pathlib import Path
from time import perf_counter
import json

import joblib
import optuna
import pandas as pd
from optuna.distributions import (
    CategoricalDistribution,
    FloatDistribution,
    IntDistribution,
)
from optuna.samplers import TPESampler
from optuna_integration.sklearn import OptunaSearchCV

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from src.utils.features import split_features

In [2]:
DATA_DIR = Path("../datasets/final/ml")
SEARCH_METHOD = "optuna"
MODEL_DIR = Path("../models") / SEARCH_METHOD
RESULTS_DIR = Path("../results/model_training") / SEARCH_METHOD

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 1
N_SPLITS = 5
N_TRIALS = 30
# trials sequenciais garantem que o TPE use todo o histórico anterior e seja reproduzível.
SEARCH_N_JOBS = 1
#o paralelismo fica dentro dos estimadores que oferecem suporte.
ESTIMATOR_N_JOBS = -1

TRAINING_CONFIGS = {
    "cicids2017": ["cicids2017"],
    "unsw_nb15": ["unsw_nb15"],
    "iot23": ["iot23"],
    "cicids2017_unsw_nb15": ["cicids2017", "unsw_nb15"],
    "cicids2017_iot23": ["cicids2017", "iot23"],
    "unsw_nb15_iot23": ["unsw_nb15", "iot23"],
}

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [3]:
reference_df = pd.read_parquet(DATA_DIR / "cicids2017_train.parquet")
numerical_features, categorical_features = split_features(reference_df)
feature_columns = numerical_features + categorical_features
del reference_df

def build_pipeline(estimator):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", StandardScaler(), numerical_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )
    return Pipeline(steps=[("preprocessor", preprocessor), ("classifier", estimator)])

In [4]:
def categorical(values):
    return CategoricalDistribution(values)


MODEL_CONFIGS = {
    "decision_tree": {
        "estimator": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "param_distributions": {
            "classifier__max_depth": categorical([None, 3, 5, 8, 10, 15, 20, 30]),
            "classifier__min_samples_split": IntDistribution(2, 20),
            "classifier__min_samples_leaf": IntDistribution(1, 10),
            "classifier__class_weight": categorical([None, "balanced"]),
        },
    },
    "random_forest": {
        "estimator": RandomForestClassifier(
            random_state=RANDOM_STATE, n_jobs=ESTIMATOR_N_JOBS
        ),
        "param_distributions": {
            "classifier__n_estimators": IntDistribution(100, 300),
            "classifier__max_depth": categorical([None, 10, 15, 20, 30]),
            "classifier__min_samples_leaf": IntDistribution(1, 10),
            "classifier__max_features": categorical(["sqrt", "log2", 0.5, 1.0]),
            "classifier__class_weight": categorical(
                [None, "balanced", "balanced_subsample"]
            ),
        },
    },
    "xgboost": {
        "estimator": XGBClassifier(
            random_state=RANDOM_STATE,
            n_jobs=ESTIMATOR_N_JOBS,
            eval_metric="logloss",
            tree_method="hist",
        ),
        "param_distributions": {
            "classifier__n_estimators": IntDistribution(100, 300),
            "classifier__max_depth": IntDistribution(3, 8),
            "classifier__learning_rate": FloatDistribution(0.03, 0.3, log=True),
            "classifier__subsample": FloatDistribution(0.7, 1.0),
            "classifier__colsample_bytree": FloatDistribution(0.7, 1.0),
        },
    },
    "linear_svm": {
        "estimator": LinearSVC(random_state=RANDOM_STATE),
        "param_distributions": {
            "classifier__C": FloatDistribution(1e-3, 1e2, log=True),
            "classifier__class_weight": categorical([None, "balanced"]),
        },
    },
    "mlp": {
        "estimator": MLPClassifier(random_state=RANDOM_STATE),
        "param_distributions": {
            "classifier__hidden_layer_sizes": categorical([(50,), (100,), (50, 50), (100, 50)]),
            "classifier__alpha": FloatDistribution(1e-5, 1e-2, log=True),
            "classifier__learning_rate_init": FloatDistribution(1e-4, 1e-2, log=True),
            "classifier__early_stopping": categorical([True, False]),
        },
    },
}

C:\Users\valde\AppData\Local\Temp\ipykernel_18900\2254232033.py:2: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (50,) which is of type tuple.
  return CategoricalDistribution(values)
C:\Users\valde\AppData\Local\Temp\ipykernel_18900\2254232033.py:2: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (100,) which is of type tuple.
  return CategoricalDistribution(values)
C:\Users\valde\AppData\Local\Temp\ipykernel_18900\2254232033.py:2: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (50, 50) which is of type tuple.
  return CategoricalDistribution(values)
C:\Users\valde\AppData\Local\Temp\ipykernel_18900\2254232033.py:2: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float

In [5]:
#modelos com orçamento de 30 trials.
MODELS_TO_RUN = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]

cross_validation = StratifiedKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)

scoring = {
    "precision_attack": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_attack": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1_attack": make_scorer(f1_score, pos_label=1, zero_division=0),
    "precision_benign": make_scorer(precision_score, pos_label=0, zero_division=0),
    "recall_benign": make_scorer(recall_score, pos_label=0, zero_division=0),
    "f1_benign": make_scorer(f1_score, pos_label=0, zero_division=0),
    "f1_macro": make_scorer(
        f1_score, average="macro", labels=[0, 1], zero_division=0
    ),
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
}

In [6]:
def load_training_data(dataset_names):
    columns = feature_columns + ["label_binary", "dataset_name"]
    dataframes = [
        pd.read_parquet(DATA_DIR / f"{dataset_name}_train.parquet", columns=columns)
        for dataset_name in dataset_names
    ]
    training_df = pd.concat(dataframes, ignore_index=True)
    features = training_df[feature_columns]
    target = training_df["label_binary"].eq("ATTACK").astype("int8")
    stratification_labels = (
        training_df["dataset_name"].astype("string")
        + "__"
        + target.astype("string")
    )
    cv_splits = list(cross_validation.split(features, stratification_labels))
    return features, target, cv_splits


def serialize_params(params):
    return json.dumps(params, sort_keys=True, default=lambda value: value.item())

In [7]:
def train_and_select_model(model_name, training_name, dataset_names):
    features, target, cv_splits = load_training_data(dataset_names)
    model_config = MODEL_CONFIGS[model_name]
    model_dir = MODEL_DIR / model_name
    results_dir = RESULTS_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE),
        study_name=f"{model_name}_{training_name}",
    )
    search = OptunaSearchCV(
        estimator=build_pipeline(model_config["estimator"]),
        param_distributions=model_config["param_distributions"],
        cv=cv_splits,
        scoring=scoring["f1_macro"],
        n_trials=N_TRIALS,
        study=study,
        refit=True,
        n_jobs=SEARCH_N_JOBS,
        random_state=RANDOM_STATE,
        verbose=1,
        error_score="raise",
        return_train_score=False,
    )

    search_start = perf_counter()
    search.fit(features, target)
    search_elapsed_seconds = perf_counter() - search_start

    candidate_results = pd.DataFrame(search.cv_results_)
    candidate_results = candidate_results.rename(columns={
        **{f"split{fold}_test_score": f"split{fold}_test_f1_macro" for fold in range(N_SPLITS)},
        "mean_test_score": "mean_test_f1_macro",
        "std_test_score": "std_test_f1_macro",
    })
    completed_trials = sorted(
        (trial for trial in study.trials if trial.state.name == "COMPLETE"),
        key=lambda trial: trial.number,
    )
    if len(completed_trials) != len(candidate_results):
        raise RuntimeError("Quantidade de trials concluídos incompatível com cv_results_.")
    candidate_results.insert(0, "trial_number", [trial.number for trial in completed_trials])
    candidate_results.insert(1, "trial_state", [trial.state.name for trial in completed_trials])
    candidate_results.insert(2, "trial_duration_seconds", [
        trial.duration.total_seconds() if trial.duration is not None else None
        for trial in completed_trials
    ])
    candidate_results.insert(3, "params", [
        serialize_params(trial.params) for trial in completed_trials
    ])
    candidate_results["rank_test_f1_macro"] = (
        candidate_results["mean_test_f1_macro"]
        .rank(method="min", ascending=False)
        .astype("int64")
    )
    candidate_results = candidate_results.sort_values("rank_test_f1_macro")
    candidate_results.insert(0, "random_state", RANDOM_STATE)
    candidate_results.insert(0, "n_splits", N_SPLITS)
    candidate_results.insert(0, "n_trials", N_TRIALS)
    candidate_results.insert(0, "training_rows", len(features))
    candidate_results.insert(0, "training_source", training_name)
    candidate_results.insert(0, "search_method", SEARCH_METHOD)
    candidate_results.insert(0, "model", model_name)
    candidate_results.to_csv(results_dir / f"{training_name}_trials.csv", index=False)

    # validação cruzada adicional da configuração vencedora para calcular,
    # em uma única execução, as médias e os desvios-padrão de todas as métricas,
    # pois o OptunaSearchCV aceita apenas uma métrica de scoring por busca.
    winner_cv_n_jobs = (
        1 if model_name in {"random_forest", "xgboost"} else -1
    )
    winner_cv_start = perf_counter()
    winner_cv_results = cross_validate(
        clone(search.best_estimator_),
        features,
        target,
        scoring=scoring,
        cv=cv_splits,
        n_jobs=winner_cv_n_jobs,
        return_train_score=False,
        error_score="raise",
    )
    winner_cv_elapsed_seconds = perf_counter() - winner_cv_start

    fold_results = pd.DataFrame({
        "model": model_name,
        "search_method": SEARCH_METHOD,
        "training_source": training_name,
        "training_rows": len(features),
        "n_trials": N_TRIALS,
        "n_splits": N_SPLITS,
        "random_state": RANDOM_STATE,
        "fold": range(1, N_SPLITS + 1),
        "fit_time_seconds": winner_cv_results["fit_time"],
        "score_time_seconds": winner_cv_results["score_time"],
    })
    for metric in scoring:
        fold_results[metric] = winner_cv_results[f"test_{metric}"]
    fold_results.to_csv(results_dir / f"{training_name}_winner_folds.csv", index=False)

    model_path = model_dir / f"{training_name}.joblib"
    joblib.dump(search.best_estimator_, model_path, compress=3)

    total_elapsed_seconds = search_elapsed_seconds + winner_cv_elapsed_seconds
    result = {
        "model": model_name,
        "search_method": SEARCH_METHOD,
        "training_source": training_name,
        "training_rows": len(features),
        "n_trials": N_TRIALS,
        "n_splits": N_SPLITS,
        "random_state": RANDOM_STATE,
        "selection_metric": "f1_macro",
        "n_completed_trials": len(completed_trials),
        "params": serialize_params(search.best_params_),
        "best_search_f1_macro": search.best_score_,
        "search_elapsed_seconds": search_elapsed_seconds,
        "search_elapsed_minutes": search_elapsed_seconds / 60,
        "refit_time_seconds": search.refit_time_,
        "winner_cv_elapsed_seconds": winner_cv_elapsed_seconds,
        "winner_cv_elapsed_minutes": winner_cv_elapsed_seconds / 60,
        "elapsed_seconds": total_elapsed_seconds,
        "elapsed_minutes": total_elapsed_seconds / 60,
        "fit_time_mean": fold_results["fit_time_seconds"].mean(),
        "fit_time_std": fold_results["fit_time_seconds"].std(ddof=0),
        "score_time_mean": fold_results["score_time_seconds"].mean(),
        "score_time_std": fold_results["score_time_seconds"].std(ddof=0),
        "model_size_mb": model_path.stat().st_size / 1024 ** 2,
        "model_file": model_path.name,
    }
    for metric in scoring:
        result[f"{metric}_mean"] = fold_results[metric].mean()
        result[f"{metric}_std"] = fold_results[metric].std(ddof=0)

    display(candidate_results)
    display(fold_results)
    return result

In [8]:
all_results = []

for model_name in MODELS_TO_RUN:
    model_results = []
    for training_name, dataset_names in TRAINING_CONFIGS.items():
        print(f"Modelo: {model_name} | origem do treinamento: {training_name}")
        result = train_and_select_model(model_name, training_name, dataset_names)
        model_results.append(result)
        all_results.append(result)

    pd.DataFrame(model_results).to_csv(
        RESULTS_DIR / model_name / "summary.csv", index=False
    )

new_results_df = pd.DataFrame(all_results)
summary_path = RESULTS_DIR / "summary.csv"
if summary_path.exists():
    previous_results_df = pd.read_csv(summary_path).reindex(
        columns=new_results_df.columns
    )
    all_results_df = pd.concat(
        [previous_results_df, new_results_df], ignore_index=True
    ).drop_duplicates(subset=["model", "training_source"], keep="last")
else:
    all_results_df = new_results_df
all_results_df.to_csv(summary_path, index=False)
display(all_results_df)

Modelo: decision_tree | origem do treinamento: cicids2017


[I 2026-10-11 21:55:35,762] A new study created in memory with name: decision_tree_cicids2017
C:\Users\valde\AppData\Local\Temp\ipykernel_18900\1500463012.py:14: ExperimentalWarning: OptunaSearchCV is experimental (supported from v0.17.0). The interface can change in the future.
  search = OptunaSearchCV(
C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\optuna_integration\sklearn\sklearn.py:275: FutureWarning: `optuna.terminator` module has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  report_cross_validation_scores(trial, scores_list)
[I 2026-10-11 21:55:38,056] Trial 0 finished with value: 0.9420724068889916 and parameters: {'classifier__max_depth': 3, 'classifier__min_samples_split': 9, 'classifier__min_samples_leaf': 6, 'classifier__class_weight': 'balanced'}. Best is trial 0 with value: 0.9420724068889916.
C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\optuna_integration\skle

,model,search_method,training_source,training_rows,n_trials,n_splits,random_state,trial_number,trial_state,trial_duration_seconds,params,mean_fit_time,std_fit_time,mean_score_time,std_score_time,split0_test_f1_macro,split1_test_f1_macro,split2_test_f1_macro,split3_test_f1_macro,split4_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro
25,decision_tree,optuna,cicids2017,235490,30,5,1,25,COMPLETE,5.813928,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 2, ""classifier__min_samples_split"": 3}",1.143522,0.035127,0.016790,0.000651,0.991054,0.990759,0.991738,0.990424,0.990502,0.990896,0.000475,1
23,decision_tree,optuna,cicids2017,235490,30,5,1,23,COMPLETE,5.917538,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 2, ""classifier__min_samples_split"": 5}",1.163238,0.040349,0.017724,0.001634,0.991024,0.990798,0.991672,0.990326,0.990636,0.990891,0.000452,2
18,decision_tree,optuna,cicids2017,235490,30,5,1,18,COMPLETE,5.698785,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 3, ""classifier__min_samples_split"": 8}",1.120150,0.032854,0.017010,0.000348,0.991060,0.990093,0.992020,0.990839,0.990295,0.990861,0.000677,3
21,decision_tree,optuna,cicids2017,235490,30,5,1,21,COMPLETE,5.733552,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 3, ""classifier__min_samples_split"": 4}",1.128054,0.032373,0.016334,0.000348,0.991223,0.989826,0.992017,0.990707,0.990391,0.990833,0.000746,4
19,decision_tree,optuna,cicids2017,235490,30,5,1,19,COMPLETE,5.775047,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 2, ""classifier__min_samples_split"": 8}",1.135848,0.028530,0.016890,0.000351,0.991159,0.990639,0.991715,0.990367,0.990250,0.990826,0.000544,5
22,decision_tree,optuna,cicids2017,235490,30,5,1,22,COMPLETE,5.717615,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": 30, ""classifier__min_samples_leaf"": 3, ""classifier__min_samples_split"": 7}",1.124460,0.034325,0.016740,0.000513,0.990764,0.989760,0.992217,0.990576,0.990587,0.990781,0.000798,6
28,decision_tree,optuna,cicids2017,235490,30,5,1,28,COMPLETE,5.642639,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 5, ""classifier__min_samples_split"": 4}",1.109466,0.030781,0.016708,0.000404,0.991257,0.990414,0.991669,0.990028,0.989816,0.990637,0.000713,7
20,decision_tree,optuna,cicids2017,235490,30,5,1,20,COMPLETE,5.883894,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 1, ""classifier__min_samples_split"": 8}",1.157569,0.040185,0.016893,0.000542,0.990883,0.990360,0.991843,0.990253,0.989742,0.990616,0.000712,8
17,decision_tree,optuna,cicids2017,235490,30,5,1,17,COMPLETE,5.742822,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 4, ""classifier__min_samples_split"": 12}",1.129408,0.034785,0.016827,0.000561,0.990544,0.989901,0.991893,0.990684,0.989883,0.990581,0.000733,9
10,decision_tree,optuna,cicids2017,235490,30,5,1,10,COMPLETE,6.302211,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 2, ""classifier__min_samples_split"": 19}",1.229908,0.071814,0.020060,0.003640,0.990878,0.989576,0.991498,0.990154,0.989974,0.990416,0.000686,10


,model,search_method,training_source,training_rows,n_trials,n_splits,random_state,fold,fit_time_seconds,score_time_seconds,precision_attack,recall_attack,f1_attack,precision_benign,recall_benign,f1_benign,f1_macro,accuracy,balanced_accuracy
0,decision_tree,optuna,cicids2017,235490,30,5,1,1,1.652865,0.058292,0.979154,0.992356,0.985711,0.998083,0.994719,0.996398,0.991054,0.994246,0.993537
1,decision_tree,optuna,cicids2017,235490,30,5,1,2,1.704455,0.050690,0.978432,0.992144,0.985240,0.998029,0.994533,0.996278,0.990759,0.994055,0.993338
2,decision_tree,optuna,cicids2017,235490,30,5,1,3,1.674994,0.052639,0.981820,0.991826,0.986798,0.997951,0.995408,0.996678,0.991738,0.994692,0.993617
3,decision_tree,optuna,cicids2017,235490,30,5,1,4,1.790989,0.046051,0.978612,0.990870,0.984703,0.997710,0.994586,0.996146,0.990424,0.993843,0.992728
4,decision_tree,optuna,cicids2017,235490,30,5,1,5,1.658315,0.062228,0.977116,0.992675,0.984834,0.998161,0.994188,0.996171,0.990502,0.993885,0.993431


Modelo: decision_tree | origem do treinamento: unsw_nb15


...

Modelo: mlp | origem do treinamento: unsw_nb15_iot23


[I 2026-10-12 16:19:00,695] A new study created in memory with name: mlp_unsw_nb15_iot23
C:\Users\valde\AppData\Local\Temp\ipykernel_18900\1500463012.py:14: ExperimentalWarning: OptunaSearchCV is experimental (supported from v0.17.0). The interface can change in the future.
  search = OptunaSearchCV(
C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the 

,model,search_method,training_source,training_rows,n_trials,n_splits,random_state,trial_number,trial_state,trial_duration_seconds,params,mean_fit_time,std_fit_time,mean_score_time,std_score_time,split0_test_f1_macro,split1_test_f1_macro,split2_test_f1_macro,split3_test_f1_macro,split4_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro
26,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,26,COMPLETE,861.899120,"{""classifier__alpha"": 5.9276550057310775e-05, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.0012247884215016809}",172.300361,19.201066,0.076088,0.000422,0.987814,0.988034,0.987164,0.987246,0.987533,0.987558,0.000330,1
16,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,16,COMPLETE,804.390071,"{""classifier__alpha"": 0.00014032253527391072, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.0011992765734976515}",160.798715,11.107416,0.075402,0.000595,0.987777,0.988100,0.986393,0.986939,0.987044,0.987251,0.000612,2
17,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,17,COMPLETE,872.655900,"{""classifier__alpha"": 0.0001607937524392821, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.000702265245646305}",174.451331,28.026665,0.076437,0.000742,0.987193,0.988104,0.987181,0.986311,0.987235,0.987205,0.000567,3
14,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,14,COMPLETE,662.522143,"{""classifier__alpha"": 3.256112690445746e-05, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.0018998137387656493}",132.425505,19.158467,0.075318,0.007110,0.986761,0.987684,0.987276,0.986740,0.987246,0.987141,0.000355,4
12,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,12,COMPLETE,750.305061,"{""classifier__alpha"": 2.3938152498123657e-05, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.0015530817719481575}",149.981978,17.150993,0.075336,0.000997,0.986767,0.987697,0.987004,0.987313,0.986764,0.987109,0.000356,5
25,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,25,COMPLETE,831.855937,"{""classifier__alpha"": 0.0005135178799730877, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.0008568360429358294}",166.292122,38.069946,0.075348,0.000501,0.987085,0.987660,0.987535,0.986698,0.986549,0.987105,0.000440,6
20,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,20,COMPLETE,864.256914,"{""classifier__alpha"": 5.9657131173950535e-05, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.0005103768944370768}",172.773480,16.704137,0.074111,0.000375,0.987231,0.987637,0.987104,0.986323,0.987074,0.987074,0.000426,7
15,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,15,COMPLETE,607.400635,"{""classifier__alpha"": 1.4799397827112744e-05, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.002237897978596909}",121.401092,28.861659,0.075314,0.002107,0.986784,0.987384,0.986194,0.987206,0.987272,0.986968,0.000437,8
27,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,27,COMPLETE,535.936696,"{""classifier__alpha"": 0.00010381567794947139, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"": [50, 50], ""classifier__learning_rate_init"": 0.002097399883265077}",107.119792,2.708780,0.063922,0.007409,0.987632,0.987484,0.986871,0.985993,0.986436,0.986883,0.000619,9
13,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,13,COMPLETE,979.687099,"{""classifier__alpha"": 1.0093906831204727e-05, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.0003679441801609784}",195.863861,39.075967,0.070170,0.001014,0.987376,0.987548,0.985514,0.985747,0.987225,0.986682,0.0

,model,search_method,training_source,training_rows,n_trials,n_splits,random_state,fold,fit_time_seconds,score_time_seconds,precision_attack,recall_attack,f1_attack,precision_benign,recall_benign,f1_benign,f1_macro,accuracy,balanced_accuracy
0,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,1,193.556817,0.168751,0.970647,0.988268,0.979378,0.997054,0.992529,0.994786,0.987082,0.991677,0.990399
1,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,2,150.039937,0.178663,0.969787,0.989967,0.979773,0.997479,0.992290,0.994878,0.987325,0.991826,0.991129
2,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,3,206.301040,0.171448,0.968837,0.991773,0.980171,0.997931,0.992025,0.994969,0.987570,0.991974,0.991899
3,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,4,182.654548,0.186594,0.969269,0.987739,0.978417,0.996920,0.992170,0.994540,0.986478,0.991284,0.989955
4,mlp,optuna,unsw_nb15_iot23,470980,30,5,1,5,212.923010,0.147230,0.968158,0.992516,0.980186,0.998117,0.991839,0.994968,0.987577,0.991974,0.992177


,model,search_method,training_source,training_rows,n_trials,n_splits,random_state,selection_metric,n_completed_trials,params,best_search_f1_macro,search_elapsed_seconds,search_elapsed_minutes,refit_time_seconds,winner_cv_elapsed_seconds,winner_cv_elapsed_minutes,elapsed_seconds,elapsed_minutes,fit_time_mean,fit_time_std,score_time_mean,score_time_std,model_size_mb,model_file,precision_attack_mean,precision_attack_std,recall_attack_mean,recall_attack_std,f1_attack_mean,f1_attack_std,precision_benign_mean,precision_benign_std,recall_benign_mean,recall_benign_std,f1_benign_mean,f1_benign_std,f1_macro_mean,f1_macro_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std
6,decision_tree,optuna,cicids2017,235490,30,5,1,f1_macro,30,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 2, ""classifier__min_samples_split"": 3}",0.990896,143.680751,2.394679,1.549246,9.368995,0.156150,153.049746,2.550829,1.696324,0.050628,0.053980,0.005694,0.033790,cicids2017.joblib,0.979027,0.001549,0.991974,0.000617,0.985457,0.000757,0.997987,0.000154,0.994687,0.000401,0.996334,0.000194,0.990896,0.000475,0.994144,0.000309,0.993330,0.000316
7,decision_tree,optuna,unsw_nb15,235490,30,5,1,f1_macro,30,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": 15, ""classifier__min_samples_leaf"": 3, ""classifier__min_samples_split"": 18}",0.980032,143.332926,2.388882,1.627599,4.417783,0.073630,147.750709,2.462512,1.858751,0.135670,0.052590,0.003624,0.029778,unsw_nb15.joblib,0.940737,0.002212,0.997516,0.000773,0.968292,0.000865,0.999370,0.000196,0.984288,0.000635,0.991771,0.000240,0.980032,0.000553,0.986934,0.000377,0.990902,0.000178
8,decision_tree,optuna,iot23,235490,30,5,1,f1_macro,30,"{""classifier__class_weight"": null, ""classifier__max_depth"": 20, ""classifier__min_samples_leaf"": 4, ""classifier__min_samples_split"": 12}",0.997766,55.247233,0.920787,0.475219,2.428261,0.040471,57.675494,0.961258,0.504956,0.073266,0.055242,0.008261,0.009499,iot23.joblib,0.999402,0.000109,0.993460,0.000984,0.996422,0.000530,0.998368,0.000245,0.999851,0.000027,0.999109,0.000132,0.997766,0.000331,0.998573,0.000211,0.996656,0.000500
9,decision_tree,optuna,cicids2017_unsw_nb15,470980,30,5,1,f1_macro,30,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 7, ""classifier__min_samples_split"": 7}",0.983458,359.276696,5.987945,3.943406,11.108456,0.185141,370.385152,6.173086,4.007051,0.184049,0.088188,0.008211,0.088208,cicids2017_unsw_nb15.joblib,0.953878,0.002057,0.994310,0.000931,0.973673,0.001317,0.998562,0.000235,0.987980,0.000557,0.993243,0.000344,0.983458,0.000830,0.989246,0.000545,0.991145,0.000628
10,decision_tree,optuna,cicids2017_iot23,470980,30,5,1,f1_macro,30,"{""classifier__class_weight"": null, ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 3, ""classifier__min_samples_split"": 7}",0.993603,263.926728,4.398779,2.712662,5.547860,0.092464,269.474588,4.491243,3.231499,0.145445,0.081097,0.004668,0.035351,cicids2017_iot23.joblib,0.989206,0.000604,0.990329,0.000457,0.989767,0.000255,0.997582,0.000114,0.997298,0.000154,0.997440,0.000064,0.993603,0.000160,0.995904,0.000103,0.993813,0.000195
11,decision_tree,optuna,unsw_nb15_iot23,470980,30,5,1,f1_macro,30,"{""classifier__class_weight"": null, ""classifier__max_depth"": 15, ""classifier__min_samples_leaf"": 7, ""classifier__min_samples_split"": 6}",0.988461,237.190942,3.953182,2.327475,5.842801,0.097380,243.033744,4.050562,2.334577,0.135788,0.080396,0.003067,0.029451,unsw_nb15_iot23.joblib,0.970931,0.000876,0.992484,0.000398,0.981589,0.000561,0.998110,0.000100,0.992571,0.000230,0.995333,0.000144,0.988461,0.000353,0.992554,0.000229,0.992528,0.000269
12,random_forest,optuna,cicids2017,235490,30,5,1,f1_macro,30,"{""classifier__class_weight"": ""balanced_subsample"", ""classifier__max_depth"": null, ""classifier__max_features"": ""sqrt"", ""classifier__mi